In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
file_directory = os.getcwd()

In [3]:
main_df = pd.read_csv(file_directory + "/data/main.csv")
driver_cost_df = pd.read_csv(file_directory + "/data//driver_cost.csv")
fuel_cost_df = pd.read_csv(file_directory + "/data//fuel_cost.csv")
fuel_usage_df = pd.read_csv(file_directory + "/data//fuel_usage.csv")
service_repair_cost_df = pd.read_csv(file_directory + "/data//service_repair_cost.csv", header=None)

In [4]:
main_df.columns = main_df.columns.str.strip()
driver_cost_df.columns = driver_cost_df.columns.str.strip()
fuel_cost_df.columns = fuel_cost_df.columns.str.strip()
fuel_usage_df.columns = fuel_usage_df.columns.str.strip()

In [5]:
service_repair_cost_df = service_repair_cost_df[2:]

In [6]:
main_df["Service"] = ""
main_df["Repair"] = ""
main_df["Number of Drivers"] = 0
main_df["Driver Cost"] = 0.0
main_df["Fuel Usage"] = ""
main_df["Fuel Cost"] = 0.0
main_df["Actual Financial Cost"] = 0.0

In [7]:
date = pd.to_datetime("2025-01-01")
for idx, row in service_repair_cost_df.iterrows():
    values = row[3:].tolist()
    values = [(values[i], values[i+1]) for i in range(0, len(values), 2)]
    for idx2, value in enumerate(values):
        main_df.loc[(main_df["Vehicle Number"] == row[0]) & (main_df["Date"] == (date+pd.DateOffset(months=idx2)).strftime('%Y-%m-%d')), "Service"] = value[0]
        main_df.loc[(main_df["Vehicle Number"] == row[0]) & (main_df["Date"] == (date+pd.DateOffset(months=idx2)).strftime('%Y-%m-%d')), "Repair"] = value[1]

In [8]:
for idx3, row2 in driver_cost_df.iterrows():
    main_df.loc[main_df["Vehicle Number"] == row2["Vehicle Number"], "Number of Drivers"] = row2["Number of Drivers"]
    main_df.loc[main_df["Vehicle Number"] == row2["Vehicle Number"], "Driver Cost"] = row2["Total salary"]/12

In [9]:
date = pd.to_datetime("2025-01-01")
for idx4, row3 in fuel_usage_df.iterrows():
    values2 = row3["25-Jan":].tolist()
    for idx5, value2 in enumerate(values2):
        main_df.loc[(main_df["Vehicle Number"] == row3["Vehicle Number"]) & (main_df["Date"] == (date+pd.DateOffset(months=idx5)).strftime('%Y-%m-%d')), "Fuel Usage"] = value2

In [10]:
date = pd.to_datetime("2025-01-01")
for idx6, row4 in fuel_cost_df.iterrows():
    values3 = row4["Jan":].tolist()
    for idx7, value3 in enumerate(values3):
        main_df.loc[(main_df["Vehicle Number"] == row4["Vehicle Number"]) & (main_df["Date"] == (date+pd.DateOffset(months=idx7)).strftime('%Y-%m-%d')), "Fuel Cost"] = value3

In [11]:
date = pd.to_datetime("2025-01-01")

for idx8 in range(1, 13):

    financial_df = pd.read_csv(file_directory + f"/data/financial_{idx8}.csv")
    financial_df.columns = financial_df.columns.str.strip()

    if "Fuel Pumped Amount" in financial_df.columns:
        financial_df["Fuel Pumped Amount"] =  financial_df["Fuel Pumped Amount"].str.replace(',', '').str.strip()
        financial_df["Fuel Pumped Amount"] = financial_df["Fuel Pumped Amount"].replace('', np.nan)
        financial_df["Fuel Pumped Amount"] = financial_df["Fuel Pumped Amount"].replace('-', np.nan)
        financial_df["Fuel Pumped Amount"] =  pd.to_numeric(financial_df["Fuel Pumped Amount"], errors='coerce')
        financial_df["Fuel Pumped Amount"] = financial_df["Fuel Pumped Amount"].fillna(0)

    financial_df["Final Payment with out VAT"] =  financial_df["Final Payment with out VAT"].str.replace(',', '').str.strip()
    financial_df["Final Payment with out VAT"] = financial_df["Final Payment with out VAT"].replace('', np.nan)
    financial_df["Final Payment with out VAT"] = financial_df["Final Payment with out VAT"].replace('-', np.nan)
    financial_df["Final Payment with out VAT"] =  pd.to_numeric(financial_df["Final Payment with out VAT"], errors='coerce')
    financial_df["Final Payment with out VAT"] = financial_df["Final Payment with out VAT"].fillna(0)

    for idx9, row5 in financial_df.iterrows():
        main_df.loc[(main_df["Vehicle Number"] == row5["Vehicle Number"]) & (main_df["Date"] == (date+pd.DateOffset(months=idx8-1)).strftime('%Y-%m-%d')), "Actual Financial Cost"] = (row5["Fuel Pumped Amount"]/118*100)+row5["Final Payment with out VAT"] if "Fuel Pumped Amount" in financial_df.columns else row5["Final Payment with out VAT"]

In [12]:
main_df["Service"] =  main_df["Service"].str.replace(',', '').str.strip()
main_df["Service"] = main_df["Service"].replace('', np.nan)
main_df["Service"] = main_df["Service"].replace('-', np.nan)
main_df["Service"] =  pd.to_numeric(main_df["Service"], errors='coerce')
main_df["Service"] = main_df["Service"].fillna(0)

main_df["Repair"] =  main_df["Repair"].str.replace(',', '').str.strip()
main_df["Repair"] = main_df["Repair"].replace('', np.nan)
main_df["Repair"] = main_df["Repair"].replace('-', np.nan)
main_df["Repair"] =  pd.to_numeric(main_df["Repair"], errors='coerce')
main_df["Repair"] = main_df["Repair"].fillna(0)

main_df["Fuel Usage"] =  main_df["Fuel Usage"].str.replace(',', '').str.strip()
main_df["Fuel Usage"] = main_df["Fuel Usage"].replace('', np.nan)
main_df["Fuel Usage"] = main_df["Fuel Usage"].replace('-', np.nan)
main_df["Fuel Usage"] =  pd.to_numeric(main_df["Fuel Usage"], errors='coerce')
main_df["Fuel Usage"] = main_df["Fuel Usage"].fillna(0)

main_df["Total KM"] =  main_df["Total KM"].str.replace(',', '').str.strip()
main_df["Total KM"] = main_df["Total KM"].replace('', np.nan)
main_df["Total KM"] = main_df["Total KM"].replace('-', np.nan)
main_df["Total KM"] =  pd.to_numeric(main_df["Total KM"], errors='coerce')
main_df["Total KM"] = main_df["Total KM"].fillna(0)

In [13]:
main_df[main_df.select_dtypes(include='float').columns] = main_df.select_dtypes(include='float').round(2)

In [14]:
main_df.to_csv(file_directory + "/data/final.csv", index=False)